# Snowflake Catalog Federation Validator

Validates that a Snowflake federated (foreign) catalog is correctly configured
and operational for Lakehouse Federation / Iceberg reads.

Includes **BCR-1935 diagnostic**: detects when Snowflake's `blob.core.windows.net`
path change breaks data reads and recommends fixes.

**Parameters:**
- `catalog_name` – name of the foreign catalog to validate

**Reference:** [BCR-1935 Workaround Guide](https://docs.google.com/document/d/1z5AZf9YZx2v0TU11X_5FpmwHTqAv2-BQshonBUEv76E)

In [ ]:
dbutils.widgets.text("catalog_name", "", "Foreign Catalog Name")
CATALOG = dbutils.widgets.get("catalog_name").strip()
assert CATALOG, "catalog_name widget must be set"

print(f"Validating catalog: {CATALOG}")

## Step 1 – Catalog Exists and is FOREIGN_CATALOG

In [ ]:
import re

catalog_info = spark.sql(f"DESCRIBE CATALOG EXTENDED `{CATALOG}`").collect()
catalog_dict = {row["info_name"]: row["info_value"] for row in catalog_info}

catalog_type = catalog_dict.get("Catalog Type", catalog_dict.get("Type", ""))
connection_name = catalog_dict.get("Connection Name", catalog_dict.get("Connection", ""))
owner = catalog_dict.get("Owner", "")
catalog_options = catalog_dict.get("Options", "")

print(f"  Catalog Type   : {catalog_type}")
print(f"  Connection     : {connection_name}")
print(f"  Owner          : {owner}")
print(f"  Options        : {catalog_options}")

assert catalog_type.upper() == "FOREIGN", (
    f"FAIL: Catalog type is '{catalog_type}', expected 'Foreign'. "
    "This is not a federated catalog."
)
print("\nPASS: Catalog is a FOREIGN catalog.")

## Step 2 – Underlying Connection

In [ ]:
conn_rows = spark.sql(f"DESCRIBE CONNECTION `{connection_name}`").collect()
conn_dict = {row["info_name"]: row["info_value"] for row in conn_rows}

conn_type = conn_dict.get("Type", "")
conn_comment = conn_dict.get("Comment", "")
conn_options = conn_dict.get("Options", "")
conn_read_only = conn_dict.get("Read-only", "")

host_match = re.search(r"host\s*->\s*([^,]+)", conn_options)
conn_host = host_match.group(1).strip() if host_match else "unknown"

print(f"  Connection     : {connection_name}")
print(f"  Type           : {conn_type}")
print(f"  Host           : {conn_host}")
print(f"  Read-only      : {conn_read_only}")
print(f"  Comment        : {conn_comment}")

assert conn_type == "SNOWFLAKE", (
    f"FAIL: Connection type is '{conn_type}', expected 'SNOWFLAKE'."
)
print("\nPASS: Snowflake connection is configured.")

## Step 3 – Schema Discovery

In [ ]:
schemas_df = spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`")
schemas = [row["databaseName"] for row in schemas_df.collect()]
print(f"  Discovered schemas ({len(schemas)}): {schemas}")

user_schemas = [s for s in schemas if s != "information_schema"]
assert len(user_schemas) > 0, (
    "FAIL: No user schemas discovered. "
    "Check that the Snowflake database has schemas and the connection credentials "
    "have permission to list them."
)
print(f"\nPASS: {len(user_schemas)} user schema(s) discovered.")

## Step 4 – Table Discovery (per schema)

In [ ]:
all_tables = []
for schema in user_schemas:
    try:
        tables_df = spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{schema}`")
        tables = [row["tableName"] for row in tables_df.collect()]
        print(f"  {CATALOG}.{schema}: {len(tables)} table(s) — {tables}")
        all_tables.extend([(schema, t) for t in tables])
    except Exception as e:
        print(f"  {CATALOG}.{schema}: ERROR listing tables — {e}")

assert len(all_tables) > 0, (
    "FAIL: No tables found in any schema. "
    "Check that the Snowflake database contains Iceberg tables and that "
    "the connection credentials can read them."
)
print(f"\nPASS: {len(all_tables)} table(s) discovered across all schemas.")

## Step 5 – Metadata Read (DESCRIBE per table)

In [ ]:
metadata_ok = []
metadata_fail = []

for schema, table in all_tables:
    fqn = f"`{CATALOG}`.`{schema}`.`{table}`"
    try:
        desc = spark.sql(f"DESCRIBE EXTENDED {fqn}").collect()
        desc_dict = {row["col_name"]: row["data_type"] for row in desc}
        provider = desc_dict.get("Provider", "unknown")
        location = desc_dict.get("Location", "N/A")
        metadata_loc = desc_dict.get("Metadata location", "N/A")
        print(f"  {fqn}")
        print(f"    Provider          : {provider}")
        print(f"    Location          : {location}")
        print(f"    Metadata location : {metadata_loc}")
        metadata_ok.append((schema, table, provider, metadata_loc))
    except Exception as e:
        print(f"  {fqn}: ERROR — {e}")
        metadata_fail.append((schema, table, str(e)))

if metadata_fail:
    print(f"\nWARN: {len(metadata_fail)} table(s) failed metadata read.")
else:
    print(f"\nPASS: Metadata readable for all {len(metadata_ok)} table(s).")

## Step 6 – Storage Access Validation

For Iceberg tables, Databricks reads data files directly from cloud storage.
This step checks that the external location and storage credential are
correctly configured by extracting the storage path from the table metadata
and verifying an external location covers it.

In [ ]:
storage_prefixes = set()
for schema, table, provider, metadata_loc in metadata_ok:
    if metadata_loc and metadata_loc.startswith("abfss://"):
        match = re.match(r"(abfss://[^/]+/)", metadata_loc)
        if match:
            storage_prefixes.add(match.group(1))

print(f"  Storage prefixes referenced by tables: {storage_prefixes}")

ext_locs_df = spark.sql("SHOW EXTERNAL LOCATIONS")
ext_locs = [(row["name"], row["url"]) for row in ext_locs_df.collect()]

for prefix in storage_prefixes:
    covered = any(prefix.startswith(url) or url.startswith(prefix) for _, url in ext_locs)
    status = "PASS" if covered else "FAIL"
    matching = [name for name, url in ext_locs if prefix.startswith(url) or url.startswith(prefix)]
    print(f"  {status}: {prefix} — covered by external location(s): {matching}")

    blob_prefix = prefix.replace(".dfs.core.windows.net", ".blob.core.windows.net")
    if blob_prefix != prefix:
        covered_blob = any(
            blob_prefix.startswith(url) or url.startswith(blob_prefix)
            for _, url in ext_locs
        )
        status_blob = "PASS" if covered_blob else "WARN (may be needed — see BCR-1935 step)"
        matching_blob = [
            name for name, url in ext_locs
            if blob_prefix.startswith(url) or url.startswith(blob_prefix)
        ]
        print(f"  {status_blob}: {blob_prefix} — covered by: {matching_blob}")

## Step 7 – BCR-1935 Diagnostic (blob vs dfs endpoint mismatch)

Snowflake **BCR-1935** (2025_03 bundle) changed Iceberg metadata to embed
`blob.core.windows.net` paths instead of `dfs.core.windows.net`.

This causes Databricks to fail reading data files because credentials are
typically configured for the `dfs` endpoint only. This step inspects the
Iceberg metadata JSON to detect the actual endpoint used in data file paths,
and checks whether a matching external location or Spark config exists.

**Reference:** [BCR-1935 Workaround Guide](https://docs.google.com/document/d/1z5AZf9YZx2v0TU11X_5FpmwHTqAv2-BQshonBUEv76E)

In [ ]:
bcr1935_affected = []
bcr1935_clean = []

for schema, table, provider, metadata_loc in metadata_ok:
    fqn = f"{CATALOG}.{schema}.{table}"
    if not metadata_loc or not metadata_loc.startswith("abfss://"):
        print(f"  SKIP: {fqn} — no abfss metadata location")
        continue

    meta_uses_blob = ".blob.core.windows.net" in metadata_loc
    meta_uses_dfs = ".dfs.core.windows.net" in metadata_loc

    acct_match = re.search(r"abfss://([^@]+)@([^.]+)\.(blob|dfs)\.core\.windows\.net", metadata_loc)
    container = acct_match.group(1) if acct_match else "unknown"
    storage_account = acct_match.group(2) if acct_match else "unknown"
    meta_endpoint = acct_match.group(3) if acct_match else "unknown"

    # Try to read the Iceberg metadata JSON to inspect data file paths
    data_file_endpoint = "unknown"
    try:
        meta_df = spark.read.format("text").load(metadata_loc)
        meta_text = "\n".join([row["value"] for row in meta_df.collect()])
        blob_refs = re.findall(r"@[^.]+\.blob\.core\.windows\.net", meta_text)
        dfs_refs = re.findall(r"@[^.]+\.dfs\.core\.windows\.net", meta_text)
        if blob_refs and not dfs_refs:
            data_file_endpoint = "blob"
        elif dfs_refs and not blob_refs:
            data_file_endpoint = "dfs"
        elif blob_refs and dfs_refs:
            data_file_endpoint = "mixed (blob + dfs)"
        else:
            data_file_endpoint = "relative paths (check manifest)"
    except Exception as e:
        data_file_endpoint = f"unreadable ({str(e)[:100]})"

    print(f"  {fqn}")
    print(f"    Storage account       : {storage_account}")
    print(f"    Container             : {container}")
    print(f"    Metadata endpoint     : {meta_endpoint}")
    print(f"    Data file endpoint    : {data_file_endpoint}")

    is_affected = (meta_uses_blob or "blob" in data_file_endpoint)
    if is_affected:
        blob_prefix = f"abfss://{container}@{storage_account}.blob.core.windows.net/"
        blob_covered = any(
            blob_prefix.startswith(url) or url.startswith(blob_prefix)
            for _, url in ext_locs
        )
        has_spark_key = False
        try:
            conf_val = spark.conf.get(
                f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
                "NOT_SET"
            )
            has_spark_key = (conf_val != "NOT_SET")
        except Exception:
            pass

        workaround_ok = blob_covered or has_spark_key
        print(f"    BCR-1935 AFFECTED     : YES")
        print(f"    Blob ext location     : {'PASS' if blob_covered else 'MISSING'}")
        print(f"    Blob spark config key : {'PASS' if has_spark_key else 'NOT SET'}")
        if not workaround_ok:
            print(f"    ACTION REQUIRED       : Add external location for {blob_prefix}")
            print(f"                            OR set cluster spark config:")
            print(f"                            fs.azure.account.key.{storage_account}.blob.core.windows.net <key>")
        else:
            print(f"    Workaround            : IN PLACE")
        bcr1935_affected.append((fqn, storage_account, container, blob_covered, has_spark_key))
    else:
        print(f"    BCR-1935 AFFECTED     : No (uses dfs endpoint)")
        bcr1935_clean.append(fqn)

print()
if bcr1935_affected:
    unresolved = [t for t in bcr1935_affected if not t[3] and not t[4]]
    print(f"BCR-1935 IMPACT: {len(bcr1935_affected)} table(s) affected.")
    if unresolved:
        print(f"  {len(unresolved)} table(s) have NO workaround in place — queries will fail.")
    else:
        print("  All affected tables have a workaround configured.")
else:
    print("BCR-1935 IMPACT: None — all tables use dfs endpoint.")

## Step 8 – Data Read (SELECT per table)

The final and most important check: can we actually read data?

In [ ]:
read_ok = []
read_fail = []

for schema, table in all_tables:
    fqn = f"`{CATALOG}`.`{schema}`.`{table}`"
    try:
        df = spark.sql(f"SELECT * FROM {fqn} LIMIT 5")
        rows = df.collect()
        count = len(rows)
        print(f"  PASS: {fqn} — returned {count} row(s)")
        if count > 0:
            print(f"    Sample: {rows[0].asDict()}")
        read_ok.append((schema, table, count))
    except Exception as e:
        err = str(e)
        print(f"  FAIL: {fqn} — {err[:200]}")
        read_fail.append((schema, table, err))

## Summary

In [ ]:
bcr_status = "NOT AFFECTED"
if bcr1935_affected:
    unresolved = [t for t in bcr1935_affected if not t[3] and not t[4]]
    if unresolved:
        bcr_status = f"AFFECTED — {len(unresolved)} table(s) UNRESOLVED"
    else:
        bcr_status = f"AFFECTED — workaround in place for {len(bcr1935_affected)} table(s)"

print("=" * 70)
print(f"SNOWFLAKE FEDERATION VALIDATION SUMMARY — {CATALOG}")
print("=" * 70)
print()
print(f"  Catalog type        : {catalog_type}")
print(f"  Connection          : {connection_name} (type={conn_type})")
print(f"  Snowflake host      : {conn_host}")
print(f"  Schemas discovered  : {len(user_schemas)}")
print(f"  Tables discovered   : {len(all_tables)}")
print(f"  Metadata readable   : {len(metadata_ok)}/{len(all_tables)}")
print(f"  BCR-1935 (blob/dfs) : {bcr_status}")
print(f"  Data readable       : {len(read_ok)}/{len(all_tables)}")
print()

overall_pass = (not read_fail and not metadata_fail)

if read_fail:
    print("FAILED TABLES:")
    for schema, table, err in read_fail:
        short_err = err[:200]
        print(f"  - {CATALOG}.{schema}.{table}")
        print(f"    Error: {short_err}")
    print()

    bcr_fqns = {t[0] for t in bcr1935_affected}
    bcr_related_fails = [
        f"{CATALOG}.{s}.{t}" for s, t, _ in read_fail
        if f"{CATALOG}.{s}.{t}" in bcr_fqns
    ]
    if bcr_related_fails:
        accounts = set(t[1] for t in bcr1935_affected)
        print("BCR-1935 RELATED FAILURES:")
        for fqn in bcr_related_fails:
            print(f"  - {fqn}")
        print()
        print("FIX (choose one):")
        print()
        print("  Option A — Cluster Spark config (quick workaround):")
        for acct in accounts:
            print(f"    fs.azure.account.key.{acct}.blob.core.windows.net  <storage_account_key>")
        print("    Set in cluster Advanced Options > Spark Config, then restart.")
        print()
        print("  Option B — External location + storage credential (UC-managed, recommended):")
        for acct in accounts:
            containers = set(t[2] for t in bcr1935_affected if t[1] == acct)
            for c in containers:
                print(f"    CREATE EXTERNAL LOCATION <name>")
                print(f"      URL 'abfss://{c}@{acct}.blob.core.windows.net/'")
                print(f"      WITH (STORAGE CREDENTIAL <credential_name>);")
        print()
        print("    Ensure the storage credential has Storage Blob Data Reader role")
        print("    at the storage account level in Azure Portal.")
        print()
        print("  Reference: Snowflake BCR-1935 (2025_03 bundle)")
        print("  https://docs.google.com/document/d/1z5AZf9YZx2v0TU11X_5FpmwHTqAv2-BQshonBUEv76E")
    else:
        print("COMMON FIXES:")
        print("  1. Ensure an external location covers BOTH .dfs. and .blob. storage endpoints")
        print("  2. Verify the storage credential has Storage Blob Data Reader on the storage account")
        print("  3. Check catalog 'authorized_paths' includes all storage prefixes")
        print("  4. Confirm the Snowflake Iceberg table's external volume is accessible")

if overall_pass:
    print("ALL CHECKS PASSED — catalog federation is fully operational.")